## How to run this notebook

There are two ways. Both are safe; choose either.

**Way 1: run your own copy.** No warning, and your changes are kept.

- At the top of this page, choose **Copy to Drive** (or **File → Save a copy in Drive**).
- Your copy opens in a new tab. It is saved in your Google Drive, in the folder **Colab Notebooks**.
- In your copy, choose **Runtime → Run all**.

**Way 2: run this page directly.** Quicker, but your changes are not kept.

- Choose **Runtime → Run all**.
- Colab shows this warning:

  > Warning: This notebook was not authored by Google. This notebook is being loaded from GitHub. It may request access to your data stored with Google, or read data and credentials from other sessions. Please review the source code before executing this notebook.

- Choose **Run anyway**.

**Why it is safe.** Colab shows this warning for every notebook that does not come from Google, before it has looked at the code. This notebook does not ask for your Google account, your Drive or any other data of yours. It only saves the table it analyses, and any figures it draws, in this session, which ends when you close the page. You can read every line of the code below before you run it.

# Subscription services: one lifestyle line

**Hayashi's Quantification Method III** · a worked example from the StatNex teaching guide

**The story.** A survey asked 40 people which of 11 subscription services they pay for (1 = yes, blank = no): from music and video streaming to e-books, language apps, fitness apps and meal kits.

**The question.** Is there one underlying "lifestyle" line along which people and services can be ordered?

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `09_multivariate/quantification_iii.csv` (40 rows) as `data.csv`, the file the code reads.

In [ ]:
with open("data.csv", "w", encoding="utf-8") as file:
    file.write("""\
No.,Music streaming,Video streaming,Mobile gaming,Fitness app,News,Meal kit,Language learning,E-books,Podcasts,Cloud storage,Online courses
1.0,,,,,1.0,,,,1.0,1.0,1.0
2.0,1.0,,,,,,,,1.0,,
3.0,,,,,1.0,,,1.0,,1.0,
4.0,,,,1.0,,1.0,1.0,,,,
5.0,,,,,,,1.0,1.0,,1.0,1.0
6.0,,,,,1.0,,,1.0,,1.0,1.0
7.0,,,,1.0,1.0,1.0,,,,,
8.0,1.0,1.0,,,,,,,,,
9.0,1.0,,,,,,,,,,
10.0,,,,,1.0,,,,1.0,1.0,
11.0,,,,,,1.0,,,,,
12.0,,,1.0,,1.0,,,,1.0,1.0,
13.0,,1.0,,1.0,,,,,1.0,1.0,
14.0,,,,1.0,1.0,,1.0,,,,1.0
15.0,,1.0,1.0,,,,,,,,
16.0,,,,1.0,,1.0,1.0,1.0,,,1.0
17.0,,,,1.0,,,1.0,,,,
18.0,,,,1.0,,1.0,,,,,
19.0,,,,1.0,1.0,,1.0,1.0,1.0,1.0,
20.0,,,,1.0,1.0,,1.0,,,1.0,1.0
21.0,,,,1.0,,1.0,1.0,,,,1.0
22.0,,,1.0,,1.0,,1.0,1.0,,,1.0
23.0,,,,,1.0,,,,1.0,1.0,
24.0,,,,,,,,,,1.0,
25.0,1.0,,1.0,,1.0,,,,,,1.0
26.0,1.0,,,,1.0,,,1.0,1.0,1.0,1.0
27.0,1.0,,,,,,,,,,
28.0,,1.0,1.0,,,,,,,,
29.0,1.0,1.0,1.0,,,,,,,,
30.0,,,,,,,,,1.0,1.0,
31.0,,,1.0,1.0,,1.0,1.0,,,,1.0
32.0,,1.0,1.0,,,,,,,,
33.0,,,,,1.0,,1.0,,,,
34.0,,,,1.0,1.0,,1.0,1.0,,,1.0
35.0,1.0,1.0,,,,,,,,,
36.0,,,,,,1.0,,,,,1.0
37.0,1.0,1.0,,,,,,,1.0,1.0,
38.0,,,,,1.0,,,,,,1.0
39.0,,,,1.0,,,1.0,,,,1.0
40.0,,,,,,1.0,,,,,1.0
""")

## Libraries

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)

## The data

In [ ]:
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
variables = ["Music streaming", "Video streaming", "Mobile gaming", "Fitness app", "News", "Meal kit", "Language learning", "E-books", "Podcasts", "Cloud storage", "Online courses"]
# 0/1 data: a 1 means the case chose that column. A blank means "not chosen", so it
# is a 0 rather than a missing value.
x = data[variables].apply(pd.to_numeric, errors="coerce").fillna(0.0).to_numpy(float)
labels = list(variables)
incomplete = 0

# A case that chose nothing, and a column nobody chose, carry no information and are
# left out.
live_rows = x.sum(axis=1) > 0
live_columns = x.sum(axis=0) > 0
summary = pd.DataFrame({"n": [live_rows.sum(), (~live_rows).sum() + incomplete,
                              len(x) + incomplete]},
                       index=["Valid cases", "Incomplete cases", "Total"])
summary["share"] = summary["n"] / summary.loc["Total", "n"]
print(summary)
x = x[np.ix_(live_rows, live_columns)]
labels = [label for label, keep in zip(labels, live_columns) if keep]

## Cross-tabulation and correlation matrix

In [ ]:
# How often two columns were chosen together, and the correlation of the 0/1 columns
# (undefined, so NaN, for a column everyone chose).
crosstab = pd.DataFrame((x.T @ x).astype(int), index=labels, columns=labels)
with np.errstate(invalid="ignore", divide="ignore"):
    correlation = np.corrcoef(x, rowvar=False)
print()
print(crosstab)
print()
print(pd.DataFrame(correlation, index=labels, columns=labels).round(4))

## Eigenvalues

In [ ]:
# Quantification method III is correspondence analysis of the response table. It gives
# every case and every column a score at once, chosen so that, over all the 1s in the
# table, the score of the case and the score of the column it chose correlate as highly
# as possible. scipy, statsmodels and scikit-learn have no correspondence analysis, so
# it is written out (R's ca::ca gives the same numbers). With r and c the row and
# column totals, the eigenvalues of D_c^-1/2 X' D_r^-1 X D_c^-1/2 are the squared
# correlations. The largest is always 1, from the trivial solution "give everything
# the same score", and is dropped. Of the min(rows, columns) - 1 axes left, those
# whose eigenvalue is 0 (1e-10 or less: rounding) carry nothing and are left out too:
# with category codes each item's columns add up to 1 in every row, so (items - 1) of
# them have eigenvalue 0. An axis's contribution is its eigenvalue's share of their sum.
r, c, total = x.sum(axis=1), x.sum(axis=0), x.sum()
scaled = (x / np.sqrt(c)).T @ (x / r[:, None]) / np.sqrt(c)
values, vectors = np.linalg.eigh((scaled + scaled.T) / 2)
order = np.argsort(-values)
values, vectors = values[order], vectors[:, order]
values, vectors = values[1:min(x.shape)], vectors[:, 1:min(x.shape)]
live = values > 1e-10
eigenvalues, vectors = values[live], vectors[:, live]
axes = len(eigenvalues)
share = eigenvalues / eigenvalues.sum()
eigen = pd.DataFrame({"axis": range(1, axes + 1), "eigenvalue": eigenvalues, "share": share,
                      "cumulative": share.cumsum(), "correlation": np.sqrt(eigenvalues)})
print()
print(eigen.to_string(index=False))

## Category scores

In [ ]:
# Each axis is standardised to mean 0 and variance 1, weighting each column by how
# often it was chosen. An axis and its mirror image are the same, so each one is
# turned round if needed to make the first column's score positive.
scores = vectors / np.sqrt(c)[:, None]
scores = scores - (c @ scores) / total
scores = scores / np.sqrt((c[:, None] * scores**2).sum(axis=0) / total)
scores = scores * np.where(scores[0] < 0, -1.0, 1.0)
axis_names = [f"axis {j + 1}" for j in range(axes)]
category_scores = pd.DataFrame(scores, index=labels, columns=axis_names)
print()
print(category_scores.round(4))

## What the result says

Quantification method III orders people and items at the same time, so that people sit near the services they pay for. It needs only yes/no answers, here from 40 people about 11 services.

The eigenvalue table shows the strength of each axis. On axis 1 the correlation coefficient between the people's scores and the services' scores is 0.86, and the axis holds 28% of the pattern. Axis 2 has 0.73 and 20%.

The category scores on axis 1 place the services along a line. Video streaming (2.13), music streaming (1.96) and mobile gaming (1.20) are at one end: entertainment. Podcasts, cloud storage and news sit near the middle: information. Language learning (-0.83), fitness app (-0.84) and meal kit (-1.18) are at the other end: self-improvement and home. So yes, there is one strong lifestyle line. But 40 people are few, and nothing here shows the line would hold in another panel.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.